# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ariba86/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** K-means clustering (k=5), applied to 8 standardized behavioral features
(impressions, clicks, ranking position, and query-diversity signals).

**Why it fits my lane:** My lane is Structured Content Archetype Clustering — the goal
is to group content pages into interpretable performance categories, not predict a
single outcome. K-means is the right tool because it discovers natural groupings
directly from behavioral similarity without needing a labeled target.

In [5]:
import duckdb, os, getpass
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Token
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily':      f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d':  f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Features (Step 3 wala)
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

# Query signals (Step 4 wala)
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()
qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')

# Clean + scale
feature_cols = ['imp_last30', 'imp_prev30', 'clk_last30', 'pos_last30',
                 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
cluster_data = data.dropna(subset=feature_cols).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data[feature_cols])

# K-means (k=5, jaisa elbow method se decide kiya tha)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_data['cluster'] = kmeans.fit_predict(X_scaled)

cluster_names = {0: 'High-Traffic Decliner', 1: 'Buried / Weak', 2: 'One-Query Dependent',
                  3: 'Steady Middle', 4: 'Elite Grower'}
cluster_data['archetype'] = cluster_data['cluster'].map(cluster_names)

print(f'{len(cluster_data):,} rows ready. Cluster counts:')
print(cluster_data['cluster'].value_counts().sort_index())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,620 rows ready. Cluster counts:
cluster
0    23654
1    57733
2     1843
3    18385
4        5
Name: count, dtype: int64


In [8]:
cluster_profile = cluster_data.groupby('cluster')[feature_cols].mean().round(2)
cluster_profile['count'] = cluster_data['cluster'].value_counts().sort_index()
cluster_profile

,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_share,count
cluster,,,,,,,,,
0,472.93,676.16,2.23,16.00,3.87,0.10,0.76,0.75,23654
1,2113.02,2691.99,8.27,14.47,27.52,0.08,0.73,0.26,57733
2,33694.36,37968.15,113.86,8.91,229.41,0.03,0.59,0.18,1843
3,287.24,450.85,0.38,48.54,11.24,0.29,0.35,0.38,18385
4,450275.60,190599.60,2789.40,4.25,2574.40,0.14,0.49,0.12,5


In [9]:
cluster_names = {
    0: 'One-Query Dependent',
    1: 'Steady Middle',
    2: 'High-Traffic Decliner',
    3: 'Buried / Weak',
    4: 'Elite Grower'
}
cluster_data['archetype'] = cluster_data['cluster'].map(cluster_names)

print(cluster_data['archetype'].value_counts())

archetype
Steady Middle            57733
One-Query Dependent      23654
Buried / Weak            18385
High-Traffic Decliner     1843
Elite Grower                 5
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Validation approach:** Since this is unsupervised clustering (no ground-truth label),
there is no train/test split in the traditional sense. Instead, I validate using:
1. An internal cohesion metric (silhouette score) on a held-out sample.
2. A cross-client generalization check — confirming the clusters aren't just an
   artifact of one or two dominant clients, by checking archetype diversity per client.

In [6]:
from sklearn.metrics import silhouette_score
import numpy as np

sample_idx = np.random.RandomState(42).choice(len(X_scaled), size=10000, replace=False)
score = silhouette_score(X_scaled[sample_idx], cluster_data['cluster'].values[sample_idx])
print(f'Silhouette Score: {score:.3f}')

Silhouette Score: 0.285


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Model vs. Baseline (same data, same pages):**

| Metric | Baseline (traffic-only) | Model (K-means, 8 features) |
|---|---|---|
| Categories | 3 (High/Medium/Low traffic) | 5 (behavioral archetypes) |
| Distinguishes growth vs. decline? | No | Yes |
| Silhouette score (structure quality) | N/A (no clustering) | 0.285 |

The baseline cannot separate pages moving in opposite directions. Within its
"High Traffic" bucket alone, 5 pages were growing (Elite Grower) while 1,762
pages in the same bucket were declining (High-Traffic Decliner) — a distinction
invisible to a single-metric rule. Similarly, within "Low Traffic," 18,067 pages
were flagged by the model as One-Query Dependent (a specific structural risk)
rather than simply "weak," a nuance the baseline cannot surface.

In [10]:
# Baseline (Week-4): sirf traffic ke basis par 3 buckets
def simple_baseline(imp):
    if imp >= 10000:
        return 'High Traffic'
    elif imp >= 500:
        return 'Medium Traffic'
    else:
        return 'Low Traffic'

cluster_data['baseline_group'] = cluster_data['imp_last30'].apply(simple_baseline)

# Model: K-means (already trained)
comparison = cluster_data.groupby(['baseline_group', 'archetype']).size().unstack(fill_value=0)
print(comparison)

archetype       Buried / Weak  Elite Grower  High-Traffic Decliner  \
baseline_group                                                       
High Traffic                6             5                   1762   
Low Traffic             15920             0                      1   
Medium Traffic           2459             0                     80   

archetype       One-Query Dependent  Steady Middle  
baseline_group                                      
High Traffic                     46           2121  
Low Traffic                   18067          19121  
Medium Traffic                 5541          36491  


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Interpretation:** The five archetypes found this run:
- **Elite Grower** (5 pages) — massive scale (~450K impressions), genuine growth
  (imp_last30 > imp_prev30), best average position (~4.3).
- **High-Traffic Decliner** (1,843 pages) — high traffic (~33.7K) but declining.
- **One-Query Dependent** (23,654 pages) — ~75% of traffic from a single query.
- **Steady Middle** (57,733 pages) — the majority archetype, average performance.
- **Buried / Weak** (18,385 pages) — low traffic, worst position (~48.5).

**Errors / limitations:** Because there's no ground truth, "errors" show up as
boundary ambiguity rather than misclassification. The silhouette score (0.285)
is moderate, meaning some pages near cluster boundaries could plausibly sit in
either group — these should be spot-checked by a human, not acted on automatically.
Cluster sizes shifted somewhat from an earlier run on the same pipeline (a smaller
Elite Grower group this time), which is expected given the warehouse reflects a
rolling window of the most recent data — not a fixed, frozen snapshot. The model
does not reward complexity for its own sake — k=5 was chosen via the elbow method,
not an arbitrarily high number.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.